# MobileNetV2 - Traffic Sign Classification
- Dataset: splitdata.rar
- Model: custom MobileNetV2
- Input: 224x224


In [ ]:
# setup
# import basic packages
import os, sys, time, json, csv, random, math, warnings
from pathlib import Path
import numpy as np
from PIL import Image
warnings.filterwarnings('ignore')

# mount drive
from google.colab import drive
drive.mount('/content/drive')

# setup status
print("Setup done!")


In [ ]:
# load data
# import file tools
from pathlib import Path
import shutil, subprocess

# set archive paths
DRIVE_RAR_PATH = Path('/content/drive/MyDrive/splitdata.rar')
LOCAL_RAR_PATH = Path('/content') / DRIVE_RAR_PATH.name
EXTRACT_ROOT = Path('/content/splitdata_extracted')
CLEAR_EXTRACT_ROOT = True
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# check rar file
if not DRIVE_RAR_PATH.exists():
    print(f'Khong tim thay file rar: {DRIVE_RAR_PATH}')
    raise SystemExit

# copy rar to local colab
print(f'Tim thay file rar: {DRIVE_RAR_PATH}')
if not LOCAL_RAR_PATH.exists() or LOCAL_RAR_PATH.stat().st_size != DRIVE_RAR_PATH.stat().st_size:
    print('Copy rar ve /content...')
    shutil.copy2(DRIVE_RAR_PATH, LOCAL_RAR_PATH)

# clear old extracted data
if CLEAR_EXTRACT_ROOT and EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

# install unrar and extract
print('Cai unrar va giai nen...')
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=True)
subprocess.run(['unrar', 'x', '-o+', str(LOCAL_RAR_PATH), str(EXTRACT_ROOT) + '/'], check=True)

# helper functions
def count_images(folder):
    return sum(1 for p in Path(folder).rglob('*') if p.suffix.lower() in IMAGE_EXTS)

def valid_data_dir(folder):
    folder = Path(folder)
    return all((folder / split).is_dir() and count_images(folder / split) > 0 for split in ['train', 'val', 'test'])

# find train/val/test folder
DATA_DIR = next((p for p in [EXTRACT_ROOT, *EXTRACT_ROOT.rglob('*')] if p.is_dir() and valid_data_dir(p)), None)

# stop if data structure is wrong
if DATA_DIR is None:
    print('Khong tim thay data hop le. Can co cau truc: train/val/test, moi thu muc chua folder class.')
    raise SystemExit

# print split summary
print(f'DATA_DIR: {DATA_DIR}')
for split in ['train', 'val', 'test']:
    split_dir = DATA_DIR / split
    num_classes = len([p for p in split_dir.iterdir() if p.is_dir()])
    print(f'{split:5s}: {count_images(split_dir):,} images | {num_classes} classes')


In [ ]:
# config
# import torch and plot packages
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12

# training settings
CONFIG = {
    'img_size': 224,
    'resize_enabled': 0,    # 1 = resize in Colab | 0 = keep prepared image size
    'resize_size': 224,       # model input size; if resize_enabled=0, set this to current image size
    'augment_enabled': 1,    # 1 = augment online while training | 0 = no online augment
    'batch_size': 32,  # if Colab still OOM, reduce to 16
    'epochs': 20,
    'lr': 0.003,
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'num_classes': 12,
    'width_mult': 1.0,
    'warmup_epochs': 5,
    'label_smoothing': 0.05,
    'dropout': 0.2,
    'grad_clip': 5.0,
    'patience': 5,
    'save_every': 1,
    'seed': 42,
    'resume': False,  # resume
    'checkpoint_dir': '/content/drive/MyDrive/mobilenetv2_gtsrb/checkpoints/',
    'log_dir': '/content/drive/MyDrive/mobilenetv2_gtsrb/logs/',
}

# create output folders
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['log_dir'], exist_ok=True)

# set device and random seed
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])
    torch.backends.cudnn.deterministic = True

# print config summary
print(f" Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Config: {json.dumps(CONFIG, indent=2)}")


In [ ]:
# data loader
# import dataset tools
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

# read config values
IMG_SIZE = CONFIG.get('resize_size', CONFIG['img_size'])
RESIZE_ENABLED = bool(CONFIG.get('resize_enabled', 1))
DATA_DIR = Path(DATA_DIR)
TRAIN_SOURCE_DIR = DATA_DIR / 'train'
VAL_SOURCE_DIR = DATA_DIR / 'val'
TEST_SOURCE_DIR = DATA_DIR / 'test'

# check split folders
missing = [str(p) for p in [TRAIN_SOURCE_DIR, VAL_SOURCE_DIR, TEST_SOURCE_DIR] if not p.is_dir()]
if missing:
    print('Khong tim thay cac thu muc data sau:')
    for p in missing:
        print(f'- {p}')
    raise SystemExit

# resize helper
def maybe_resize(size, extra_pixels=0):
    if not RESIZE_ENABLED:
        return []
    target = size + extra_pixels
    return [transforms.Resize((target, target))]

# read class names
raw_train_for_classes = ImageFolder(TRAIN_SOURCE_DIR)
CLASS_NAMES = raw_train_for_classes.classes
CONFIG['num_classes'] = len(CLASS_NAMES)
NUM_CLASSES = CONFIG['num_classes']

# compute mean/std
MEAN_STD_MAX_IMAGES = 3000
stats_transform = transforms.Compose(maybe_resize(IMG_SIZE) + [transforms.ToTensor()])
stats_dataset = ImageFolder(TRAIN_SOURCE_DIR, transform=stats_transform)
if len(stats_dataset) > MEAN_STD_MAX_IMAGES:
    generator = torch.Generator().manual_seed(CONFIG['seed'])
    stats_indices = torch.randperm(len(stats_dataset), generator=generator)[:MEAN_STD_MAX_IMAGES].tolist()
    stats_dataset = Subset(stats_dataset, stats_indices)

stats_loader = DataLoader(stats_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
pixel_count = 0

for images, _ in stats_loader:
    channel_sum += images.sum(dim=[0, 2, 3])
    channel_sq_sum += (images ** 2).sum(dim=[0, 2, 3])
    pixel_count += images.size(0) * images.size(2) * images.size(3)

MEAN = (channel_sum / pixel_count).tolist()
STD = torch.sqrt(channel_sq_sum / pixel_count - torch.tensor(MEAN) ** 2).tolist()

# build train transform
AUGMENT_ENABLED = bool(CONFIG.get('augment_enabled', 0))
if AUGMENT_ENABLED:
    base_augment_steps = maybe_resize(IMG_SIZE, extra_pixels=8) + [
        transforms.RandomCrop((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(20),
        transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
        transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
        transforms.RandomGrayscale(p=0.05),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
    ]
else:
    base_augment_steps = maybe_resize(IMG_SIZE) + [transforms.ToTensor()]

# build val/test transform
augment_preview_transform = transforms.Compose(base_augment_steps)
train_transform = transforms.Compose(base_augment_steps + [transforms.Normalize(MEAN, STD)])
val_transform = transforms.Compose(maybe_resize(IMG_SIZE) + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

# create datasets
train_dataset = ImageFolder(TRAIN_SOURCE_DIR, transform=train_transform)
val_dataset = ImageFolder(VAL_SOURCE_DIR, transform=val_transform)
test_dataset = ImageFolder(TEST_SOURCE_DIR, transform=val_transform)
train_full = train_dataset
train_full_no_aug = ImageFolder(TRAIN_SOURCE_DIR, transform=val_transform)

# create dataloaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

# print dataset summary
print('\nDataset Statistics:')
print(f'DATA_DIR: {DATA_DIR}')
print(f'Classes: {NUM_CLASSES}')
print(f'Train: {len(train_dataset)} images')
print(f'Val:   {len(val_dataset)} images')
print(f'Test:  {len(test_dataset)} images')
print(f'Resize enabled: {int(RESIZE_ENABLED)}')
print(f'Input/resize size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size: {CONFIG["batch_size"]}')
print(f'MEAN: {[round(x, 4) for x in MEAN]}')
print(f'STD:  {[round(x, 4) for x in STD]}')
print('\nClass mapping:')
for idx, name in enumerate(CLASS_NAMES):
    print(f'{idx:2d}: {name}')
print(f'\nOnline augmentation enabled: {int(AUGMENT_ENABLED)}')
print('Val/test are never augmented.')


In [ ]:
# data summary
# import counter for class counts
from collections import Counter

# count labels by split
def get_label_counts(dataset):
    if hasattr(dataset, 'targets'):
        labels = dataset.targets
    else:
        labels = [dataset[i][1] for i in range(len(dataset))]
    return Counter(labels)

# collect split counts
train_counts = get_label_counts(train_dataset)
val_counts = get_label_counts(val_dataset)
test_counts = get_label_counts(test_dataset)

# print split summary
print('Split summary:')
print(f"Train: {len(train_dataset):,} images")
print(f"Val:   {len(val_dataset):,} images")
print(f"Test:  {len(test_dataset):,} images")

# print class summary table
print('\nClass summary:')
print(f"{'ID':>2} | {'Class':35s} | {'Train':>6} | {'Val':>6} | {'Test':>6}")
print('-' * 65)
for class_id, class_name in enumerate(CLASS_NAMES):
    print(f"{class_id:>2} | {class_name[:35]:35s} | {train_counts.get(class_id, 0):>6} | {val_counts.get(class_id, 0):>6} | {test_counts.get(class_id, 0):>6}")

# plot class counts
x = np.arange(NUM_CLASSES)
width = 0.26
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width, [train_counts.get(i, 0) for i in range(NUM_CLASSES)], width, label='Train', color='#3498db')
ax.bar(x, [val_counts.get(i, 0) for i in range(NUM_CLASSES)], width, label='Val', color='#2ecc71')
ax.bar(x + width, [test_counts.get(i, 0) for i in range(NUM_CLASSES)], width, label='Test', color='#f39c12')
ax.set_xlabel('Class ID')
ax.set_ylabel('Number of images')
ax.set_title('Class Distribution by Split')
ax.set_xticks(x)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'class_distribution_by_split.png'), dpi=150, bbox_inches='tight')
plt.show()

# print min/max train class
print(f"\nMin train samples: Class {min(train_counts, key=train_counts.get)} = {min(train_counts.values())}")
print(f"Max train samples: Class {max(train_counts, key=train_counts.get)} = {max(train_counts.values())}")


In [ ]:
# image preview
# preview augmentation
preview_raw_dataset = ImageFolder(TRAIN_SOURCE_DIR, transform=None)
preview_img, preview_label = preview_raw_dataset[0]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle(f'Augmentation example - class {preview_label}: {CLASS_NAMES[preview_label]}', fontsize=14, fontweight='bold')

axes = axes.flat
axes[0].imshow(preview_img)
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')

for i in range(1, 8):
    aug_img = augment_preview_transform(preview_img).permute(1, 2, 0).clamp(0, 1)
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Augment {i}', fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'augmentation_preview.png'), dpi=150, bbox_inches='tight')
plt.show()

# inverse normalize for later image display
inv_normalize = transforms.Compose([
    transforms.Normalize(mean=[-m/s for m, s in zip(MEAN, STD)],
                         std=[1/s for s in STD])
])


## MobileNetV2
### Inverted residual block
### ReLU6 + skip connection


In [ ]:
# model

def _make_divisible(v, divisor=8, min_value=None):
    """Make channels divisible."""
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


# conv block
class ConvBNReLU6(nn.Sequential):
    """Conv + BN + ReLU6."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, groups=1):
        padding = (kernel_size - 1) // 2
        super().__init__(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding,
                      groups=groups, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU6(inplace=True)
        )


# residual block
class InvertedResidual(nn.Module):
    """Inverted residual block."""
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()
        self.stride = stride
        assert stride in [1, 2]

        hidden_dim = int(round(in_channels * expand_ratio))
        self.use_skip = (stride == 1 and in_channels == out_channels)

        layers = []
        # expand channels
        if expand_ratio != 1:
            layers.append(ConvBNReLU6(in_channels, hidden_dim, kernel_size=1))

        # depthwise conv
        layers.append(ConvBNReLU6(hidden_dim, hidden_dim, kernel_size=3,
                                   stride=stride, groups=hidden_dim))

        # project channels
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, 1, 0, bias=False),
            nn.BatchNorm2d(out_channels),
        ])

        self.conv = nn.Sequential(*layers)

    # forward residual block
    def forward(self, x):
        if self.use_skip:
            return x + self.conv(x)  # skip
        else:
            return self.conv(x)


# mobilenetv2 model
class MobileNetV2(nn.Module):
    """MobileNetV2."""
    def __init__(self, num_classes=43, width_mult=1.0, dropout=0.2):
        super().__init__()

        # stage config
        inverted_residual_setting = [
            # t, c,   n, s
            [1, 16,  1, 1],
            [6, 24,  2, 2],
            [6, 32,  3, 2],
            [6, 64,  4, 2],
            [6, 96,  3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]

        # first conv layer
        input_channels = _make_divisible(32 * width_mult)
        last_channels = _make_divisible(1280 * max(1.0, width_mult))

        first_stride = 2 if IMG_SIZE >= 160 else 1
        features = [ConvBNReLU6(3, input_channels, kernel_size=3, stride=first_stride)]
        # stride by image size

        # === Inverted Residual Blocks ===
        for t, c, n, s in inverted_residual_setting:
            output_channels = _make_divisible(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(InvertedResidual(input_channels, output_channels,
                                                  stride=stride, expand_ratio=t))
                input_channels = output_channels

        # last conv layer
        features.append(ConvBNReLU6(input_channels, last_channels, kernel_size=1))

        self.features = nn.Sequential(*features)

        # === Classifier ===
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(last_channels, num_classes),
        )

        # initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    # forward residual block
    def forward(self, x):
        x = self.features(x)
        x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# create model
# clear memory
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# instantiate model
model = MobileNetV2(
    num_classes=CONFIG['num_classes'],
    width_mult=CONFIG['width_mult'],
    dropout=CONFIG['dropout']
).to(device)

# Test forward pass
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
out = model(dummy)
print("Model ready!")
print(f"Input:  {dummy.shape}")
print(f"Output: {out.shape}")

# parameters
# count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n Model Statistics:")
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Model size:       ~{total_params * 4 / 1024 / 1024:.1f} MB (FP32)")

# architecture
print("\nMobileNetV2 architecture:")
print(model)


In [ ]:
# optimizer
# create loss function
# loss
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'])

# optimizer
# create SGD optimizer
optimizer = optim.SGD(
    model.parameters(),
    lr=CONFIG['lr'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
    nesterov=True
)

# learning rate scheduler
# scheduler
class WarmupCosineScheduler:
    """Warmup cosine scheduler."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, base_lr, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        if self.current_epoch <= self.warmup_epochs:
            # warmup
            lr = self.base_lr * (self.current_epoch / self.warmup_epochs)
        else:
            # cosine
            progress = (self.current_epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + (self.base_lr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def get_lr(self):
        return self.optimizer.param_groups[0]['lr']

    def state_dict(self):
        return {'current_epoch': self.current_epoch}

    def load_state_dict(self, state_dict):
        self.current_epoch = state_dict['current_epoch']

# create scheduler
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs=CONFIG['warmup_epochs'],
    total_epochs=CONFIG['epochs'],
    base_lr=CONFIG['lr']
)

# preview learning rate curve
lrs = []
temp_sched = WarmupCosineScheduler(optimizer, CONFIG['warmup_epochs'], CONFIG['epochs'], CONFIG['lr'])
for e in range(CONFIG['epochs']):
    temp_sched.step()
    lrs.append(temp_sched.get_lr())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, CONFIG['epochs']+1), lrs, 'b-', linewidth=2)
ax.axvline(x=CONFIG['warmup_epochs'], color='r', linestyle='--', alpha=0.7, label=f'Warmup ends (epoch {CONFIG["warmup_epochs"]})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title(' Learning Rate Schedule', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# reset scheduler before training
# create scheduler
scheduler = WarmupCosineScheduler(optimizer, CONFIG['warmup_epochs'], CONFIG['epochs'], CONFIG['lr'])

print(f"Loss: CrossEntropyLoss (label_smoothing={CONFIG['label_smoothing']})")
print(f"Optimizer: SGD (lr={CONFIG['lr']}, momentum={CONFIG['momentum']}, nesterov=True)")
print(f"Scheduler: Warmup({CONFIG['warmup_epochs']}ep) + CosineAnnealing")


In [ ]:
# train loop
# import mixed precision tools
from torch.cuda.amp import GradScaler, autocast

# train one epoch
def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


# validate one epoch
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


# save latest checkpoint
def save_checkpoint(state, filepath):
    torch.save(state, filepath)


# load checkpoint
def load_checkpoint(filepath, model, optimizer, scheduler):
    if os.path.exists(filepath):
        print(f"Loading checkpoint: {filepath}")
        ckpt = torch.load(filepath, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        return ckpt.get('epoch', 0), ckpt.get('best_val_acc', 0), ckpt.get('training_log', None)
    return 0, 0, None


# prepare training log
training_log = {
    'epoch': [], 'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'lr': [], 'epoch_time': []
}

# resume or reset run
if not CONFIG['resume']:
    import shutil
    if os.path.exists(CONFIG['checkpoint_dir']):
        shutil.rmtree(CONFIG['checkpoint_dir'])
        os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
    if os.path.exists(CONFIG['log_dir']):
        shutil.rmtree(CONFIG['log_dir'])
        os.makedirs(CONFIG['log_dir'], exist_ok=True)
    print('Deleted old checkpoints and logs.')

start_epoch = 0
best_val_acc = 0
patience_counter = 0
latest_ckpt = os.path.join(CONFIG['checkpoint_dir'], 'checkpoint_latest.pth')

if CONFIG['resume'] and os.path.exists(latest_ckpt):
    start_epoch, best_val_acc, saved_log = load_checkpoint(latest_ckpt, model, optimizer, scheduler)
    if saved_log:
        training_log = saved_log
    print(f"Resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.2f}%")
else:
    print("Start new training...")

# mixed precision scaler
scaler = GradScaler()

print(f"\n{'='*80}")
print(f"{'TRAINING MOBILENETV2 - CUSTOM TRAFFIC SIGN DATA':^80}")
print(f"{'='*80}")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>10} | {'Val Loss':>10} | {'Val Acc':>10} | {'LR':>10} | {'Time':>8} | {'Status':>10}")
print(f"{'-'*80}")

# epoch loop
for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
    epoch_start = time.time()

    # update lr
    scheduler.step()
    current_lr = scheduler.get_lr()

    # train and validate
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    epoch_time = time.time() - epoch_start

    # update log
    training_log['epoch'].append(epoch)
    training_log['train_loss'].append(train_loss)
    training_log['train_acc'].append(train_acc)
    training_log['val_loss'].append(val_loss)
    training_log['val_acc'].append(val_acc)
    training_log['lr'].append(current_lr)
    training_log['epoch_time'].append(epoch_time)

    status = ""
    # best checkpoint status
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        status = "BEST"
        save_checkpoint({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc, 'training_log': training_log,
        }, os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth'))
    else:
        patience_counter += 1
        status = f"WAIT {patience_counter}/{CONFIG['patience']}"

    print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>9.2f}% | {val_loss:>10.4f} | {val_acc:>9.2f}% | {current_lr:>10.6f} | {epoch_time:>6.1f}s | {status:>10}")

    # save periodic checkpoint
    if epoch % CONFIG['save_every'] == 0:
        save_checkpoint({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc, 'training_log': training_log,
        }, os.path.join(CONFIG['checkpoint_dir'], f'checkpoint_epoch_{epoch}.pth'))
        print(f"Checkpoint saved: checkpoint_epoch_{epoch}.pth")

    # save latest checkpoint
    save_checkpoint({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_acc': best_val_acc, 'training_log': training_log,
    }, latest_ckpt)

    # save training log
    with open(os.path.join(CONFIG['log_dir'], 'training_log.json'), 'w') as f:
        json.dump(training_log, f, indent=2)

    with open(os.path.join(CONFIG['log_dir'], 'training_history.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr', 'epoch_time'])
        for i in range(len(training_log['epoch'])):
            writer.writerow([training_log[k][i] for k in training_log])

    # early stopping
    if patience_counter >= CONFIG['patience']:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {CONFIG['patience']} epochs)")
        break

print(f"\n{'='*80}")
print(f"Training done! Best Val Accuracy: {best_val_acc:.2f}%")
print(f"Best model: {os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth')}")
total_time = sum(training_log['epoch_time'])
print(f"Total time: {total_time/60:.1f} minutes")


In [ ]:
# training and validation plots
# prepare history arrays
epochs_range = np.array(training_log['epoch'])
train_loss_arr = np.array(training_log['train_loss'])
val_loss_arr = np.array(training_log['val_loss'])
train_acc_arr = np.array(training_log['train_acc'])
val_acc_arr = np.array(training_log['val_acc'])
lr_arr = np.array(training_log['lr'])
time_arr = np.array(training_log['epoch_time'])
acc_gap = train_acc_arr - val_acc_arr
loss_gap = val_loss_arr - train_loss_arr
best_epoch = int(epochs_range[np.argmax(val_acc_arr)])

# create plot canvas
fig, axes = plt.subplots(3, 2, figsize=(15, 14))
fig.suptitle('Training and Validation Diagnostics', fontsize=16, fontweight='bold')

# plot train/val loss
ax = axes[0, 0]
ax.plot(epochs_range, train_loss_arr, 'b-', label='Train Loss', linewidth=2)
ax.plot(epochs_range, val_loss_arr, 'r-', label='Val Loss', linewidth=2)
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.5, label=f'Best epoch {best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Loss Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# plot train/val accuracy
ax = axes[0, 1]
ax.plot(epochs_range, train_acc_arr, 'b-', label='Train Acc', linewidth=2)
ax.plot(epochs_range, val_acc_arr, 'r-', label='Val Acc', linewidth=2)
ax.axhline(y=best_val_acc, color='green', linestyle='--', alpha=0.5, label=f'Best {best_val_acc:.2f}%')
ax.axvline(best_epoch, color='green', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# plot accuracy gap
ax = axes[1, 0]
ax.plot(epochs_range, acc_gap, color='#8e44ad', linewidth=2)
ax.axhline(0, color='black', linewidth=1, alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Train Acc - Val Acc (%)')
ax.set_title('Accuracy Gap')
ax.grid(True, alpha=0.3)

# plot loss gap
ax = axes[1, 1]
ax.plot(epochs_range, loss_gap, color='#d35400', linewidth=2)
ax.axhline(0, color='black', linewidth=1, alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Loss - Train Loss')
ax.set_title('Loss Gap')
ax.grid(True, alpha=0.3)

# plot learning rate
ax = axes[2, 0]
ax.plot(epochs_range, lr_arr, color='#27ae60', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule')
ax.grid(True, alpha=0.3)

# plot epoch time
ax = axes[2, 1]
ax.bar(epochs_range, time_arr, color='#34495e', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Time (seconds)')
ax.set_title('Training Time per Epoch')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'training_validation_diagnostics.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved plot: {CONFIG['log_dir']}training_validation_diagnostics.png")


In [ ]:
# test
# import metrics
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score, accuracy_score)

# load best checkpoint
best_ckpt = os.path.join(CONFIG['checkpoint_dir'], 'best_model.pth')
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best model (val_acc={ckpt['best_val_acc']:.2f}%)")

# predict test set
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# calculate overall metrics
test_acc = accuracy_score(all_labels, all_preds) * 100
test_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0) * 100
test_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0) * 100
test_f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0) * 100

print(f"\n{'='*50}")
print(f"{'TEST SET RESULTS':^50}")
print(f"{'='*50}")
print(f"Accuracy:  {test_acc:.2f}%")
print(f"Precision: {test_precision:.2f}%")
print(f"Recall:    {test_recall:.2f}%")
print(f"  F1-Score:  {test_f1:.2f}%")
print(f"{'='*50}")

# print classification report
print(f"\n Classification Report:")
print(classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# metric plot
# collect overall scores
# draw overall metric bar chart
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [test_acc, test_precision, test_recall, test_f1]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
bars = ax.bar(metrics, values, color=colors, width=0.6, edgecolor='white', linewidth=2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 105)
ax.set_ylabel('Score (%)')
ax.set_title(' Overall Test Metrics', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
# save overall metric plot
plt.savefig(os.path.join(CONFIG['log_dir'], 'overall_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# class metrics
# import heatmap tool
import seaborn as sns

# build confusion matrix
cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=range(NUM_CLASSES), yticklabels=range(NUM_CLASSES))
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix - Test Accuracy: {test_acc:.2f}%')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# plot normalized confusion matrix
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm * 100, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax,
            xticklabels=range(NUM_CLASSES), yticklabels=range(NUM_CLASSES))
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Normalized Confusion Matrix (%)')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'confusion_matrix_normalized.png'), dpi=150, bbox_inches='tight')
plt.show()

# create class report folder
CLASS_REPORT_DIR = os.path.join(CONFIG['log_dir'], 'class_reports')
os.makedirs(CLASS_REPORT_DIR, exist_ok=True)

# calculate metrics per class
per_class_stats = []
for class_id, class_name in enumerate(CLASS_NAMES):
    true_mask = all_labels == class_id
    pred_mask = all_preds == class_id
    support = int(true_mask.sum())
    pred_count = int(pred_mask.sum())
    tp = int((true_mask & pred_mask).sum())
    fp = int((~true_mask & pred_mask).sum())
    fn = int((true_mask & ~pred_mask).sum())

    acc = 100.0 * tp / support if support else 0.0
    precision = 100.0 * tp / pred_count if pred_count else 0.0
    recall = acc
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    per_class_stats.append({
        'id': class_id,
        'name': class_name,
        'support': support,
        'pred_count': pred_count,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

# print class metric table
print('\nPer-class metrics:')
print(f"{'ID':>2} | {'Class':35s} | {'N':>4} | {'Acc':>6} | {'Prec':>6} | {'Rec':>6} | {'F1':>6} | {'FP':>3} | {'FN':>3}")
print('-' * 92)
for s in per_class_stats:
    print(f"{s['id']:>2} | {s['name'][:35]:35s} | {s['support']:>4} | {s['accuracy']:>5.1f}% | {s['precision']:>5.1f}% | {s['recall']:>5.1f}% | {s['f1']:>5.1f}% | {s['fp']:>3} | {s['fn']:>3}")

# plot per-class metric overview
class_ids = [s['id'] for s in per_class_stats]
acc_values = [s['accuracy'] for s in per_class_stats]
prec_values = [s['precision'] for s in per_class_stats]
rec_values = [s['recall'] for s in per_class_stats]
f1_values = [s['f1'] for s in per_class_stats]

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(class_ids, acc_values, marker='o', label='Accuracy', linewidth=2)
axes[0].plot(class_ids, f1_values, marker='o', label='F1', linewidth=2)
axes[0].set_ylabel('Score (%)')
axes[0].set_title('Per-Class Accuracy and F1')
axes[0].set_ylim(0, 105)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(class_ids, prec_values, marker='o', label='Precision', linewidth=2)
axes[1].plot(class_ids, rec_values, marker='o', label='Recall', linewidth=2)
axes[1].set_xlabel('Class ID')
axes[1].set_ylabel('Score (%)')
axes[1].set_title('Per-Class Precision and Recall')
axes[1].set_ylim(0, 105)
axes[1].set_xticks(class_ids)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'per_class_metric_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

# draw one metric chart per class
for s in per_class_stats:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle(f"Class {s['id']}: {s['name']}", fontsize=13, fontweight='bold')

    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
    metric_values = [s['accuracy'], s['precision'], s['recall'], s['f1']]
    metric_colors = ['#2ecc71' if v >= 95 else '#f39c12' if v >= 85 else '#e74c3c' for v in metric_values]
    axes[0].bar(metric_names, metric_values, color=metric_colors)
    axes[0].set_ylim(0, 105)
    axes[0].set_ylabel('Score (%)')
    axes[0].set_title('Metrics')
    axes[0].grid(True, axis='y', alpha=0.3)
    for j, v in enumerate(metric_values):
        axes[0].text(j, min(v + 2, 102), f'{v:.1f}%', ha='center', fontsize=9)

    count_names = ['Correct', 'Missed', 'False Pos']
    count_values = [s['tp'], s['fn'], s['fp']]
    axes[1].bar(count_names, count_values, color=['#2ecc71', '#e74c3c', '#3498db'])
    axes[1].set_ylabel('Images')
    axes[1].set_title(f"Support={s['support']} | Predicted={s['pred_count']}")
    axes[1].grid(True, axis='y', alpha=0.3)
    for j, v in enumerate(count_values):
        axes[1].text(j, v + max(count_values + [1]) * 0.03, str(v), ha='center', fontsize=9)

    plt.tight_layout()
    out_path = os.path.join(CLASS_REPORT_DIR, f"class_{s['id']:02d}_metrics.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

# print weakest classes
sorted_stats = sorted(per_class_stats, key=lambda x: x['accuracy'])
print(f"\nClass reports saved to: {CLASS_REPORT_DIR}")
print('\nLowest accuracy classes:')
for s in sorted_stats[:5]:
    print(f"Class {s['id']:2d} | {s['name']:35s} | acc={s['accuracy']:.1f}% | fp={s['fp']} | fn={s['fn']}")


In [ ]:
# class samples
# create sample output folder
CLASS_SAMPLE_DIR = os.path.join(CONFIG['log_dir'], 'class_samples')
os.makedirs(CLASS_SAMPLE_DIR, exist_ok=True)

# draw sample grid per class
for class_id, class_name in enumerate(CLASS_NAMES):
    # collect test images for this class
    class_indices = np.where(all_labels == class_id)[0]
    if len(class_indices) == 0:
        print(f'Class {class_id}: no test images')
        continue

    # pick up to 8 images
    sample_count = min(8, len(class_indices))
    pick_positions = np.linspace(0, len(class_indices) - 1, sample_count, dtype=int)
    sample_indices = class_indices[pick_positions]

    # draw image grid
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle(f"Class {class_id}: {class_name}", fontsize=13, fontweight='bold')

    for ax, idx in zip(axes.flat, sample_indices):
        img, _ = test_dataset[idx]
        img = inv_normalize(img).permute(1, 2, 0).clamp(0, 1)
        # get prediction and confidence
        pred_id = int(all_preds[idx])
        conf = float(all_probs[idx][pred_id]) * 100
        ok = pred_id == class_id

        ax.imshow(img)
        ax.set_title(f"Pred: {pred_id}\nConf: {conf:.1f}%", fontsize=9,
                     color='green' if ok else 'red')
        ax.axis('off')

    for ax in list(axes.flat)[sample_count:]:
        ax.axis('off')

    # save class sample grid
    plt.tight_layout()
    out_path = os.path.join(CLASS_SAMPLE_DIR, f"class_{class_id:02d}_samples.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()

print(f"Class samples saved to: {CLASS_SAMPLE_DIR}")


In [ ]:
# confidence plots
# collect confidence values
top_conf = all_probs.max(axis=1) * 100
is_correct = all_preds == all_labels
correct_conf = top_conf[is_correct]
wrong_conf = top_conf[~is_correct]

# plot confidence distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correct_conf, bins=20, alpha=0.75, label='Correct', color='#2ecc71')
if len(wrong_conf) > 0:
    ax.hist(wrong_conf, bins=20, alpha=0.75, label='Wrong', color='#e74c3c')
ax.set_xlabel('Prediction confidence (%)')
ax.set_ylabel('Number of images')
ax.set_title('Test Confidence Distribution')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'test_confidence_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

# calculate mean confidence per class
mean_conf_by_class = []
wrong_count_by_class = []
for class_id in range(NUM_CLASSES):
    class_mask = all_labels == class_id
    mean_conf = top_conf[class_mask].mean() if class_mask.sum() else 0
    wrong_count = int((class_mask & ~is_correct).sum())
    mean_conf_by_class.append(mean_conf)
    wrong_count_by_class.append(wrong_count)

# plot class confidence and errors
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(range(NUM_CLASSES), mean_conf_by_class, color='#3498db', alpha=0.8, label='Mean confidence')
ax1.set_xlabel('Class ID')
ax1.set_ylabel('Mean confidence (%)')
ax1.set_ylim(0, 105)
ax1.grid(True, axis='y', alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(range(NUM_CLASSES), wrong_count_by_class, color='#e74c3c', marker='o', linewidth=2, label='Wrong count')
ax2.set_ylabel('Wrong predictions')

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper right')
ax1.set_title('Mean Confidence and Wrong Predictions by Class')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'class_confidence_errors.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# export
# print export header
print("=" * 60)
print(f"{'MODEL ANALYSIS':^60}")
print("=" * 60)

# count model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = total_params * 4 / 1024 / 1024

print(f"\n Parameter Count:")
print(f"Total:     {total_params:>12,}")
print(f"Trainable: {trainable_params:>12,}")
print(f"Model size: {model_size_mb:.1f} MB (FP32)")

# count parameters per layer
print(f"\n Parameters per Layer:")
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"{name:20s}: {params:>10,} params")

# save final model
final_path = os.path.join(CONFIG['checkpoint_dir'], 'mobilenetv2_gtsrb_final.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'test_accuracy': test_acc,
    'test_f1': test_f1,
    'class_names': CLASS_NAMES,
    'num_classes': CONFIG['num_classes'],
}, final_path)
print(f"\n Final model saved: {final_path}")
print(f"File size: {os.path.getsize(final_path) / 1024 / 1024:.1f} MB")

# list checkpoint files
print(f"\n Checkpoint files:")
for f in sorted(os.listdir(CONFIG['checkpoint_dir'])):
    fp = os.path.join(CONFIG['checkpoint_dir'], f)
    print(f"{f:40s} ({os.path.getsize(fp)/1024/1024:.1f} MB)")

print(f"\n{'='*60}")
print("DONE. MobileNetV2 trained and evaluated.")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test F1-Score: {test_f1:.2f}%")
print(f"Model saved at: {CONFIG['checkpoint_dir']}")
print(f"Logs saved at:  {CONFIG['log_dir']}")
print(f"{'='*60}")


In [ ]:
# random test samples
# import random picker
import random

# pick random test images
random.seed(None)
test_indices = random.sample(range(len(test_dataset)), 10)

# draw prediction grid
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
fig.suptitle('Random Test Samples', fontsize=16, fontweight='bold')

correct_count = 0

for i, (ax, idx) in enumerate(zip(axes.flat, test_indices)):
    img, true_label = test_dataset[idx]

    # predict each image
    model.eval()
    with torch.no_grad():
        output = model(img.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1)
        confidence, pred_label = probs.max(1)
        pred_label = pred_label.item()
        confidence = confidence.item() * 100

    # display image
    img_show = inv_normalize(img).permute(1, 2, 0).clamp(0, 1)
    ax.imshow(img_show)

    is_correct = pred_label == true_label
    if is_correct:
        correct_count += 1
        color = 'green'
        symbol = ''
    else:
        color = 'red'
        symbol = ''

    ax.set_title(f'{symbol} Pred: {pred_label} ({confidence:.0f}%)\nTrue: {true_label}',
                 fontsize=10, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['log_dir'], 'random_test_10.png'), dpi=150, bbox_inches='tight')
plt.show()

# print sample summary
print(f"\nResult: {correct_count}/10 correct ({correct_count*10}%)")
print("\nDetails:")
for i, idx in enumerate(test_indices):
    _, true_label = test_dataset[idx]
    img, _ = test_dataset[idx]
    with torch.no_grad():
        output = model(img.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1)
        conf, pred = probs.max(1)
    status = '' if pred.item() == true_label else ''
    print(f"{status} Image {i+1}: True={CLASS_NAMES[true_label]:35s} | Pred={CLASS_NAMES[pred.item()]:35s} | Conf={conf.item()*100:.1f}%")


In [ ]:
# predict external image
# import upload and display tools
from google.colab import files, output
from IPython.display import display, HTML
import base64, io

# crop UI
def get_crop_from_js(img_path):
    """Get crop box."""
    img_pil = Image.open(img_path).convert('RGB')
    w, h = img_pil.size

    # resize image for browser display
    max_display = 700
    scale = min(max_display / w, max_display / h, 1.0)
    dw, dh = int(w * scale), int(h * scale)

    # convert image to base64
    buf = io.BytesIO()
    img_pil.resize((dw, dh)).save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()

    js_code = f"""
    (async () => {{
        const img = new window.Image();
        img.src = 'data:image/png;base64,{b64}';
        await new Promise(r => img.onload = r);

        const canvas = document.createElement('canvas');
        canvas.width = {dw}; canvas.height = {dh};
        canvas.style.cursor = 'crosshair';
        canvas.style.border = '2px solid #333';
        canvas.style.borderRadius = '8px';

        const ctx = canvas.getContext('2d');
        ctx.drawImage(img, 0, 0);

        const info = document.createElement('div');
        info.style.cssText = 'font:14px monospace;margin:8px 0;color:#555';
        info.textContent = 'Keo chuot chon vung bien bao, hoac bam OK de dung toan bo anh';

        const btnRow = document.createElement('div');
        btnRow.style.cssText = 'margin:8px 0;display:flex;gap:10px';

        const btnOK = document.createElement('button');
        btnOK.textContent = 'OK - Predict';
        btnOK.style.cssText = 'padding:8px 24px;font-size:15px;background:#2ecc71;color:#fff;border:none;border-radius:6px;cursor:pointer';

        const btnReset = document.createElement('button');
        btnReset.textContent = 'Reset';
        btnReset.style.cssText = 'padding:8px 24px;font-size:15px;background:#3498db;color:#fff;border:none;border-radius:6px;cursor:pointer';

        btnRow.appendChild(btnOK);
        btnRow.appendChild(btnReset);

        const container = document.createElement('div');
        container.appendChild(info);
        container.appendChild(canvas);
        container.appendChild(btnRow);
        document.querySelector('#output-area').appendChild(container);

        let startX=0, startY=0, drawing=false, hasRect=false;
        let rx=0, ry=0, rw=0, rh=0;

        function redraw() {{
            ctx.drawImage(img, 0, 0);
            if (hasRect) {{
                ctx.strokeStyle = '#00ff00';
                ctx.lineWidth = 3;
                ctx.setLineDash([]);
                ctx.strokeRect(rx, ry, rw, rh);
                ctx.fillStyle = 'rgba(0,255,0,0.1)';
                ctx.fillRect(rx, ry, rw, rh);
                info.textContent = 'Da chon vung ' + Math.round(rx/{scale}) + ',' + Math.round(ry/{scale}) + ' -> ' + Math.round((rx+rw)/{scale}) + ',' + Math.round((ry+rh)/{scale}) + ' | Bam OK de du doan';
            }}
        }}

        canvas.onmousedown = e => {{
            const rect = canvas.getBoundingClientRect();
            startX = e.clientX - rect.left;
            startY = e.clientY - rect.top;
            drawing = true; hasRect = false;
        }};
        canvas.onmousemove = e => {{
            if (!drawing) return;
            const rect = canvas.getBoundingClientRect();
            const mx = e.clientX - rect.left;
            const my = e.clientY - rect.top;
            rx = Math.min(startX, mx); ry = Math.min(startY, my);
            rw = Math.abs(mx - startX); rh = Math.abs(my - startY);
            hasRect = true;
            redraw();
        }};
        canvas.onmouseup = () => {{ drawing = false; }};

        btnReset.onclick = () => {{
            hasRect = false; redraw();
            info.textContent = 'Keo chuot chon vung bien bao, hoac bam OK de dung toan bo anh';
        }};

        const result = await new Promise(resolve => {{
            btnOK.onclick = () => {{
                container.remove();
                if (hasRect && rw > 5 && rh > 5) {{
                    resolve([
                        Math.round(rx / {scale}),
                        Math.round(ry / {scale}),
                        Math.round((rx + rw) / {scale}),
                        Math.round((ry + rh) / {scale})
                    ]);
                }} else {{
                    resolve(null);
                }}
            }};
        }});
        return result;
    }})()
    """
    crop = output.eval_js(js_code)
    return crop

def predict_and_show(img_path, crop_box=None):
    img_pil = Image.open(img_path).convert('RGB')
    img_input = img_pil.crop(crop_box) if crop_box else img_pil

    predict_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    tensor = predict_tf(img_input).unsqueeze(0).to(device)

    # upload and predict model
    model.eval()
    with torch.no_grad():
        out = model(tensor)
        probs = torch.softmax(out, dim=1)
        top_p, top_i = probs.topk(5, dim=1)

    top_p = top_p[0].cpu().numpy() * 100
    top_i = top_i[0].cpu().numpy()

    # draw input and result
    fig, axes = plt.subplots(1, 3, figsize=(16, 5),
                              gridspec_kw={'width_ratios': [1, 1, 1.5]})
    axes[0].imshow(img_pil)
    if crop_box:
        import matplotlib.patches as patches
        l,t,r,b = crop_box
        rect = patches.Rectangle((l,t), r-l, b-t, linewidth=3, edgecolor='lime', facecolor='none')
        axes[0].add_patch(rect)
    axes[0].set_title(f'Original ({img_pil.size[0]}x{img_pil.size[1]})', fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(img_input)
    axes[1].set_title(f'Input -> {IMG_SIZE}x{IMG_SIZE}', fontsize=11)
    axes[1].axis('off')

    colors = ['#2ecc71' if j==0 else '#3498db' for j in range(5)]
    axes[2].barh(range(4,-1,-1), top_p, color=colors)
    axes[2].set_yticks(range(4,-1,-1))
    axes[2].set_yticklabels([CLASS_NAMES[j] for j in top_i], fontsize=9)
    axes[2].set_xlabel('Confidence (%)')
    axes[2].set_title('Top-5 Predictions', fontweight='bold')
    axes[2].set_xlim(0, 110)
    for idx, p in enumerate(top_p):
        axes[2].text(p+1, 4-idx, f'{p:.1f}%', va='center', fontsize=10)

    plt.suptitle(f' {CLASS_NAMES[top_i[0]]} ({top_p[0]:.1f}%)',
                 fontsize=14, fontweight='bold', color='green')
    plt.tight_layout()
    plt.show()
    print(f"{CLASS_NAMES[top_i[0]]} ({top_p[0]:.1f}%)")

# upload and predict
print("Upload traffic sign image:")
uploaded = files.upload()
for fname in uploaded:
    fpath = f'/content/{fname}'
    with open(fpath, 'wb') as f:
        f.write(uploaded[fname])
    print(f"\n{fname} - crop or press OK:")
    crop = get_crop_from_js(fpath)
    if crop:
        print(f"Crop: {crop}")
    else:
        print("No crop - use full image")
    predict_and_show(fpath, crop)
